In [2]:
from typing import Any

from pydantic import BaseModel
from unstructured.partition.pdf import partition_pdf

/home/biguser/codeFactory/llm_knowledge_management/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
images_path = './images'
raw_pdf_elements = partition_pdf(
    # filename="weekly_market_recap.pdf",
    filename="Wi-Fi 6(802.11ax)解析23：QTP（Quiet Time Period）和QP.pdf",
    strategy="hi_res",
    extract_images_in_pdf=True,
    extract_image_block_types=["Image", "Figure"],
    infer_table_structure=True,
    include_metadata=True,
    include_page_breaks=True,
    # chunking_strategy="by_title",
    max_characters=1500,         # 降低每個段落的最大字元數
    new_after_n_chars=1400,      # 降低觸發新段落的字元數
    combine_text_under_n_chars=500,  # 降低合併門檻，使分段更細
    extract_image_block_output_dir=images_path,
)

# for i in raw_pdf_elements:
#     print(i.category)

In [4]:
category_counts = {}

for element in raw_pdf_elements:
    category = str(type(element))
    if category in category_counts:
        category_counts[category] += 1
    else:
        category_counts[category] = 1

unique_categories = set(category_counts.keys())
category_counts

{"<class 'unstructured.documents.elements.Header'>": 1,
 "<class 'unstructured.documents.elements.Title'>": 5,
 "<class 'unstructured.documents.elements.Image'>": 5,
 "<class 'unstructured.documents.elements.NarrativeText'>": 19,
 "<class 'unstructured.documents.elements.PageBreak'>": 4,
 "<class 'unstructured.documents.elements.Table'>": 2}

In [5]:
# Image summarizer

import base64
import os

from langchain.chat_models import ChatOpenAI
from langchain.schema.messages import HumanMessage

class ImageSummarizer:

    def __init__(self, image_path) -> None:
        self.image_path = image_path
        self.prompt = """
You are an assistant tasked with summarizing images for retrieval.
These summaries will be embedded and used to retrieve the raw image.
Give a concise summary of the image that is well optimized for retrieval.
"""

    def base64_encode_image(self):
        with open(self.image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode("utf-8")

    def summarize(self, prompt = None):
        base64_image_data = self.base64_encode_image()
        chat = ChatOpenAI(model="gpt-4o-mini", max_tokens=1000)

        # gpt4 vision api doc - https://platform.openai.com/docs/guides/vision
        response = chat.invoke(
            [
                HumanMessage(
                    content=[
                        {
                            "type": "text",
                            "text": prompt if prompt else self.prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/jpeg;base64,{base64_image_data}"},
                        },
                    ]
                )
            ]
        )
        return base64_image_data, response.content

In [6]:
image_data_list = []
image_summary_list = []

for img_file in sorted(os.listdir(images_path)):
    if img_file.endswith(".jpg"):
        summarizer = ImageSummarizer(os.path.join(images_path, img_file))
        data, summary = summarizer.summarize()
        image_data_list.append(data)
        image_summary_list.append(summary)

/tmp/ipykernel_2947195/1072906220.py:25: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  chat = ChatOpenAI(model="gpt-4o-mini", max_tokens=1000)


In [7]:
image_summary_list

['A digital avatar with long silver hair, wearing a bear-themed hat, and a stylish checkered outfit. The avatar has large, expressive brown eyes and is holding a decorative item resembling a feather or bone.',
 'Diagram illustrating a data structure layout with fields for Element ID, Length, Quiet count, Quiet period, Quiet duration, and Quiet offset, along with their respective byte sizes.',
 'Diagram illustrating a QTP (Quick Transfer Protocol) scenario in a wireless communication system. It shows the interaction between various devices: an HE STA (Station) making a QTP request, an HE AP (Access Point) responding, and multiple HE STAs engaged in QTP setup. Includes details on P2P (Peer-to-Peer) transmission and conditions for immediate responses.',
 'Diagram illustrating the QTP (Quiet Time Period) setup mechanism in wireless communication, detailing interactions between an Access Point (AP) and multiple stations (STA 1, STA 2, STA 3, STA 4). Key elements include QTP requests, respon

In [11]:
class Element(BaseModel):
    type: str
    text: Any

table_elements = []
text_elements = []
for element in raw_pdf_elements:
    if "unstructured.documents.elements.Table" in str(type(element)):
        table_elements.append(Element(type="table", text=str(element)))
    else:
        print(element)
        # text_elements.append(Element(type="text", text=str(element)))

https://zhuanlan.zhihu.com/p/110772486
Wi-Fi 6(802.11ax)解析 23：QTP（Quiet Time Period）和 QP

Wi-Fi 研習者
Wi-Fi 話題下的優秀答主
15 人贊同了該文章
序言
QTP（Quiet Time Period）是 802.11ax 中新引入的技術，而 QP（Quiet Period）是 802.11 協議初始就有的一個技術，兩者都是在 802.11 協議中比較冷門的一個技術。在 筆者的認知中，這兩個技術最大的用處就是為了相容，畢竟是商業協定，各種場景都需 要考慮到。所以本文僅僅做一個筆記記錄下。
通道靜默 QP（Quiet Period）
這裡 Quiet Period 我們翻譯成通道靜默，這是參考 802.11 權威指南的翻譯。我們知道 802.11 協議是工作在 2.4GHz 和 5GHz 頻段的，這個頻段在很多國家是有雷達設備也工 作的。為了避免對雷達的干擾，802.11 協議中採用了 TPC 和 DFS 兩項技術。然而，究 竟周邊有沒有雷達設備呢，這個時候實際上要 802.11 設備做一個探測的。
但是由於無線網路傳輸本身會影響探測結果。此時，我們就需要把整個網路靜默一段時 間，也就是不允許任何的資料包傳輸，讓網路去檢測周圍有沒有雷達信號。這個機制就
是 QP（Quite Period）機制。
QP 時間是一個週期的時間，該週期間隔是通過 Beacon 或者 Probe Response 中的 Quiet Element 元素進行安排的。而 QP 時間的保護是通過 NAV 機制來實現的，這裡是 通過 Quiet Element 來實現 NAV，有點類似於 CFP 時間可以由 CF Parameter Set 裡 面元素設置的一樣。
bytes 1 1 1 1 2 2 Element ID Quiet Quiet duration Quiet offset period 40

Quiet Element 的結構如上圖所示，其中 Quiet Period 是靜默期，代表了 QP 時間的間 隔（單位為幾個 Beacon 間隔，也就是 TBTT 時間），Quiet Duration 為靜默持續的 Duration 時間。Quiet Offset 是一個時間偏移，一般而言，Q

In [9]:
table_elements

[Element(type='table', text='AP Re ae QTP Setup toSTA1 EEKOD) QTP P2P Frame STA1 Request to STA3 (11ax, P2P) STA 2 Quiet Time Period (HE NAV) (11ax) STA 3 Quiet Time Period (HE (11ax, P2P) Se en eh er ee STA 4 (Legacy)'),
 Element(type='table', text='AP agp Response toSTA1 QTP Request P2P Frame to STA3 STA1 (11ax, P2P)} Quiet Time Period (NAV) STA 2 (11ax) (11ax, P2P) STA 4 Quiet Time Period get “ire (Legacy)')]

In [10]:
text_elements

[Element(type='text', text='https://zhuanlan.zhihu.com/p/110772486'),
 Element(type='text', text='Wi-Fi 6(802.11ax)解析 23：QTP（Quiet Time Period）和 QP'),
 Element(type='text', text=''),
 Element(type='text', text='Wi-Fi 研習者'),
 Element(type='text', text='Wi-Fi 話題下的優秀答主'),
 Element(type='text', text='15 人贊同了該文章'),
 Element(type='text', text='序言'),
 Element(type='text', text='QTP（Quiet Time Period）是 802.11ax 中新引入的技術，而 QP（Quiet Period）是 802.11 協議初始就有的一個技術，兩者都是在 802.11 協議中比較冷門的一個技術。在 筆者的認知中，這兩個技術最大的用處就是為了相容，畢竟是商業協定，各種場景都需 要考慮到。所以本文僅僅做一個筆記記錄下。'),
 Element(type='text', text='通道靜默 QP（Quiet Period）'),
 Element(type='text', text='這裡 Quiet Period 我們翻譯成通道靜默，這是參考 802.11 權威指南的翻譯。我們知道 802.11 協議是工作在 2.4GHz 和 5GHz 頻段的，這個頻段在很多國家是有雷達設備也工 作的。為了避免對雷達的干擾，802.11 協議中採用了 TPC 和 DFS 兩項技術。然而，究 竟周邊有沒有雷達設備呢，這個時候實際上要 802.11 設備做一個探測的。'),
 Element(type='text', text='但是由於無線網路傳輸本身會影響探測結果。此時，我們就需要把整個網路靜默一段時 間，也就是不允許任何的資料包傳輸，讓網路去檢測周圍有沒有雷達信號。這個機制就'),
 Element(type='text', text='是 QP（Quite Period）機制。'),
 Element(type='text', 

In [22]:
print(len(table_elements))
print(len(text_elements))

2
34


In [26]:
table_elements[1]

Element(type='table', text='AP agp Response toSTA1 QTP Request P2P Frame to STA3 STA1 (11ax, P2P)} Quiet Time Period (NAV) STA 2 (11ax) (11ax, P2P) STA 4 Quiet Time Period get “ire (Legacy)')

In [27]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

In [28]:
prompt_text = """
  You are responsible for concisely summarizing table or text chunk:

  {element}
"""
prompt = ChatPromptTemplate.from_template(prompt_text)
summarize_chain = {"element": lambda x: x} | prompt | ChatOpenAI(temperature=0, model="gpt-4") | StrOutputParser()

/tmp/ipykernel_2473644/1321421087.py:7: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  summarize_chain = {"element": lambda x: x} | prompt | ChatOpenAI(temperature=0, model="gpt-4") | StrOutputParser()


In [29]:
tables = [i.text for i in table_elements]
table_summaries = summarize_chain.batch(tables, {"max_concurrency": 5})

texts = [i.text for i in text_elements]
text_summaries = summarize_chain.batch(texts, {"max_concurrency": 5})

In [30]:
table_summaries

['The text provides a variety of financial data. The NASDAQ is at 18196, with a variety of other statistics provided. Fixed income yields for U.S. Aggregate, U.S. Corporates, Municipals (10yr), and High Yield are given, along with their respective changes. Currency exchange rates for $ per €, $ per £, and ¥ per $ are also provided. Key rates for 2-yr, 10-yr, and 30-yr U.S. Treasuries, 10-yr German Bund, SOFR, 3-mo. EURIBOR, 6-mo. CD rate, 30-yr fixed mortgage, and Prime Rate are listed. Commodity prices for oil (WTI), gasoline, natural gas, gold, silver, copper, and corn are given. The BBG Index is at 255.17.']

In [ ]:
import uuid

from langchain.embeddings import OpenAIEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.schema.document import Document
from langchain.storage import InMemoryStore
from langchain.vectorstores import Chroma

id_key = "doc_id"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings()),
    docstore=InMemoryStore(),
    id_key=id_key,
)

# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]
summary_texts = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(text_summaries)
]
retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]
summary_tables = [
    Document(page_content=s, metadata={id_key: table_ids[i]})
    for i, s in enumerate(table_summaries)
]
retriever.vectorstore.add_documents(summary_tables)
retriever.docstore.mset(list(zip(table_ids, tables)))

# Add images
image_data_list = []
image_summary_list = []
doc_ids = [str(uuid.uuid4()) for _ in image_data_list]
summary_images = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(image_summary_list)
]
retriever.vectorstore.add_documents(summary_images)
retriever.docstore.mset(list(zip(doc_ids, image_data_list)))